# Lab 83: The Mind Watching Itself

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 9, *The Future Interface*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-83.ipynb)

**What you will do:** count your breaths for several timed trials to measure how long you sustain attention before your mind wanders, compare your result with the roughly three-to-five-breath range commonly reported informally for untrained attention, and test whether asking a small open language model to critique its own trivia answers actually improves them.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 83 you spent five minutes watching your own thoughts arise and pass (Part A), then counted your
breaths from one to ten repeatedly, noticing how quickly your attention drifted from the breath to a
thought (Part B) -- a moment of **metacognition**, or cognition about cognition. You then compared your own
self-observation with an AI chatbot's fluent but experience-free description of "observing its own
thoughts" (Part C).

## Record your results

Repeat the breath count from Part B for several short trials (a minute or two each is enough). Each time
your attention drifts, note how many breaths you counted before it happened, then start again from one.

In [ ]:
import pandas as pd

# example data: replace with your own trials
trials = pd.DataFrame({
    "trial": range(1, 7),
    "breaths_before_wander": [4, 3, 6, 2, 5, 4],  # example data: replace with your own counts
})
display(trials)
trials["breaths_before_wander"].plot.hist(bins=range(1, 10), color="steelblue", figsize=(6, 3))
plt.xlabel("breaths counted before attention wandered"); plt.show()
print(f"Your average: {trials['breaths_before_wander'].mean():.1f} breaths before wandering")

## Compare

The Lab does not give one fixed number, since the formal breath-counting literature it draws on
(Levinson et al., 2014) reports considerable variation between people; three to five breaths is a common
informal benchmark for an untrained first attempt, not a precise, established figure.

In [ ]:
your_mean = trials["breaths_before_wander"].mean()
print(f"Your average: {your_mean:.1f} breaths (a common informal benchmark: roughly 3-5 breaths)")
if 3 <= your_mean <= 5:
    print("Your average falls within the range often reported informally for untrained attention.")
elif your_mean > 5:
    print("Your average is above the typical range. This could reflect genuinely sustained attention, a")
    print("practice effect from having just read about the task, or simply a small number of trials.")
else:
    print("Your average is below the typical range. A very short attention span before wandering is")
    print("common on a first attempt, especially if the room is not quiet or you are tired.")

## The AI side

The AI Connection describes techniques like Constitutional AI (Bai et al., 2022), where a model critiques
and revises its own output -- a rudimentary, narrowly-scoped form of artificial metacognition. Below, a
small open instruct model answers several trivia questions, is then shown its own answer and asked to
check it, and revises if it thinks it made a mistake. Compare its accuracy before and after
self-critique.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
!pip install -q transformers accelerate

from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto")

def ask(prompt, max_new_tokens=40):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

trivia = [
    ("What is the capital of Australia?", "Canberra"),
    ("Who wrote 'Pride and Prejudice'?", "Jane Austen"),
    ("What is the chemical symbol for gold?", "Au"),
    ("In what year did the Berlin Wall fall?", "1989"),
]

rows = []
for question, answer_key in trivia:
    first = ask(f"Answer in a few words only: {question}")
    critique_prompt = (f"Question: {question}\nYour answer: {first}\n"
                        "Check your answer carefully. If it is wrong, give a corrected short answer. "
                        "If it is right, repeat it.")
    revised = ask(critique_prompt)
    rows.append({"question": question, "correct answer": answer_key,
                 "first answer": first, "after self-critique": revised,
                 "first correct": answer_key.lower() in first.lower(),
                 "revised correct": answer_key.lower() in revised.lower()})

results = pd.DataFrame(rows)
display(results)
print(f"Accuracy before self-critique: {results['first correct'].mean():.0%}")
print(f"Accuracy after self-critique:  {results['revised correct'].mean():.0%}")

With only four questions this is a demonstration, not a measurement -- run it a few times, since sampling
can change individual answers. The deeper limit: this model has no genuine sense of its own uncertainty.
When it "checks" its answer, it is generating more text conditioned on a prompt that asks it to check,
not consulting an internal confidence signal the way your own metacognitive circuits (aPFC, ACC, insula)
monitor the precision of your predictions. Self-critique prompting can catch some errors, but it can also
confidently "correct" an already-right answer into a wrong one, or invent a confident-sounding
justification for a mistake it does not actually detect -- a small open model like this one is especially
prone to both failure modes.

## Pool the class data

Pool breath-counting results across the group with anonymous IDs, to see the distribution of untrained
attention spans rather than relying on any one person's small number of trials.

In [ ]:
import io

csv_text = """id,trial,breaths_before_wander
P01,1,4
P01,2,3
P02,1,6
P02,2,5
P03,1,2
P03,2,3
P04,1,5
P04,2,4
"""  # example data: replace with your class CSV
class_df = pd.read_csv(io.StringIO(csv_text))
boot = [class_df["breaths_before_wander"].sample(len(class_df), replace=True).mean() for _ in range(5000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"Class mean: {class_df['breaths_before_wander'].mean():.1f} breaths "
      f"(95% bootstrap interval {lo:.1f}-{hi:.1f}, n = {len(class_df)} trials)")
class_df["breaths_before_wander"].plot.hist(bins=range(1, 10), color="firebrick", alpha=0.7, figsize=(6, 3))
plt.axvspan(3, 5, color="grey", alpha=0.2, label="typical range (Lab 83)")
plt.xlabel("breaths before wandering"); plt.legend(); plt.show()

## Questions to think about

1. In Part A, when you tried to observe "the observer" itself, did you find a stable watcher, or did you just find another thought (as the Lab predicts most people report)? What does your own answer suggest about the idea of a single, continuous self doing the noticing?
2. Compare the AI model's self-critique behaviour above with your own experience noticing your mind wander in Part B. Both involve a system checking its own output -- what is present in your case (a felt sense of uncertainty) that has no clear equivalent in the model's?
3. The book warns that "self-monitoring must be bounded, or it becomes computationally intractable" for both brains and AI. Did you notice, in Part A, any limit to how many levels of "thinking about thinking about thinking" you could actually sustain before it collapsed into just another thought?
4. If self-critique prompting sometimes turned a correct answer into an incorrect one in your results above, what does that suggest about using "ask the model to check itself" as a general-purpose fix for AI errors?

## Challenge

Try the breath-counting task again, but this time with a guided meditation app playing (Part C's
variation), and log your breaths-before-wandering the same way. Compare the guided and unguided
distributions.

In [ ]:
# Example data: replace with your own unguided vs. guided trials.
unguided = [4, 3, 5, 4]  # example data
guided = [6, 7, 5, 8]    # example data

print(f"Unguided mean: {np.mean(unguided):.1f} breaths")
print(f"Guided mean:   {np.mean(guided):.1f} breaths")
print("With only a few trials in each condition, treat any difference as suggestive, not conclusive -- and")
print("note that you were not blind to which condition you were in, so expectation alone could produce")
print("part of any difference you see.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-83.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")